In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load



# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

<center>
<p style="font-size:20pt; font-style:bold; text-align:center">
"Car Price Prediction" 
</p>
</center>


<center><img
src="https://mir-s3-cdn-cf.behance.net/project_modules/disp/96fdb578684029.5cac4a000cece.gif" style="width:70%;height:70%;">
</center>

# Introduction
Hellooo everyone, we are together with new notebook. I started a new challenge that I will make notebooks about every type of problem (regression, classification, clustering, computer vision, NLP). Starting with **'Regression Type Problem'**. In this data set, we will try to predict car prices! Excited??

Sooo let's step on the gas!!

## Libraries ⬇
<hr>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pandas.plotting import scatter_matrix

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

from scipy import stats

import joblib

import warnings
warnings.filterwarnings('ignore')

## Importing Data 📅
<hr>

In [ ]:
df = pd.read_csv('/kaggle/input/car-price-prediction-challenge/car_price_prediction.csv')

## First Glance To Data 🔎
<hr>

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
desc = ['Price','Levy','Mileage' ,'Prod. year', 'Cylinders', 'Airbags']
df[desc].describe()

>📌 The data we imported has **19237 rows** and ***18 columns***. Seems like we don't have any null value from ***info()*** method but actually in ***Levy*** column the ***null*** values entered as ***'-'*** so we had to change them to null values. We will handle this problem soon. 

>📌 The descriptive statistics shows that the ***Prices, Levy, Mileage, Cylinders and Airbags*** probably have ***right skewed*** distribution because the means of these attributes are bigger than the medians. On the other hand, ***Prod. year*** looks like ***left skewed*** distribution.

## Explatory Data Analysis 📊 and Data Cleaning 🧹
<hr>

Let's start checking the unique values for each attributes

In [ ]:
for col in df.columns:
    print(col)
    print(df[f'{col}'].unique())
    print('*'*75)

In [ ]:
df[df['Levy'] == '-']

>📌 As I mentioned earlier we had the '-' values in the Levy column which used instead of null values. We have ***5819*** of them! We will replace them with null for now and will handle what to imput there later on.

In [ ]:
# Replacing '-' with null
df['Levy'].replace({'-':np.nan}, inplace = True)
# Converting Levy type to float
df['Levy'] = df['Levy'].astype('float64')

In [ ]:
# Replacement: Yes >> True , No >> False
df['Leather interior'].replace({'Yes': True, 'No':False}, inplace=True)

>📌 We want to convert the Engine column type to float in order to do that we need to strip the ***'turbo'*** word in them. Also, we will make ***new column*** which is ***Turbo*** , will indicate that if the car has turbo or not (True and False).

In [ ]:
# Making sure that we don miss anything so making everything lower first
df['Engine volume'] = df['Engine volume'].str.lower()

# Finding the rows which has turbo in them and assigning the results to new column Turbo. 
df['Turbo'] = df['Engine volume'].str.contains('turbo')

# Slicing engine volumes and converting type to float
df['Engine volume'] = df['Engine volume'].str.slice(0,3)
df['Engine volume'] = df['Engine volume'].astype('float64')

>📌 The similar issue with ***Mileage*** as well in order to convert the type of this column to ***integer***, we need to strip ***'km'*** in them. So let's make it.

In [ ]:
df['Mileage'] = df['Mileage'].str.strip('km')
df['Mileage'] = df['Mileage'].astype('int64')

>📌 The ***'Doors'*** column has these unique values ***'04-May', '02-Mar', '>5'*** so respectively we will replace them to ***4, 2 and 5***.

In [ ]:
df['Doors'].replace({'04-May':4, '02-Mar':2, '>5':5}, inplace=True)

Dropping the ID column which will not provide any information for our model

In [ ]:
cars = df.drop('ID', axis=1)

Let's check the last version of our dataset after our cleaning.

In [ ]:
cars.head()

>📌 There are some values they are surreal to be true in the columns such as ***Mileage*** and ***Price***. So let's check them and then we will make outlier detection to get rid of some of these values.

In [ ]:
display(cars[cars.Price == cars.Price.max()])
display(cars[cars.Price < 1000])

In [ ]:
display(cars[cars.Mileage == cars.Mileage.max()])
display(cars[cars.Mileage < 1000])

As a outlier detection algorithm, we will use ***IQR calculation*** and get rid of the values which is higher or lower than ***1.5 IQR***.

In [ ]:
def detect_outliers(df,features,thold):
    outlier_indices = []
    
    for c in features:
        # 1st quartile
        Q1 = np.percentile(df[c],25)
        # 3rd quartile
        Q3 = np.percentile(df[c],75)
        # IQR
        IQR = Q3 - Q1
        # Outlier step
        outlier_step = IQR * thold
        # Detect outlier and their indeces
        outlier_list_col = df[(df[c] < Q1 - outlier_step) | (df[c] > Q3 + outlier_step)].index
        # Store indeces
        outlier_indices.extend(outlier_list_col)
    
    
    return outlier_indices

In [ ]:
features = ['Price', 'Levy', 'Mileage']
outliers = detect_outliers(cars,features, 1.5)
deleted_df = cars.drop(cars.loc[outliers].index,axis=0)

In [ ]:
top10_cars = deleted_df['Manufacturer'].value_counts().sort_values(ascending = False)[:10]
top10_mean_prices = [deleted_df[deleted_df['Manufacturer'] == i]['Price'].mean() for i in list(top10_cars.index)]

fig = plt.figure(figsize=(14,6))
ax = fig.add_subplot(121)
sns.barplot(top10_cars.index, top10_cars.values, palette='hot')
plt.xticks(rotation = 90)
plt.ylabel('# Cars')
plt.title('Top10 The Most Frequent Cars')

ax2 = fig.add_subplot(122)
sns.lineplot(top10_cars.index, top10_mean_prices, color='r')
plt.xticks(rotation = 90)
plt.ylabel('Mean Prices')
plt.title("Top10 Cars' Mean Prices")
plt.show()

In [ ]:
deleted_df.groupby('Doors')['Drive wheels'].value_counts()

In [ ]:
dd_val = np.array(deleted_df.groupby('Doors')['Drive wheels'].value_counts().values).reshape(3,3)
dd_sum = dd_val.sum(axis=1).reshape(3,1)
dd_sum = np.c_[dd_sum, dd_sum, dd_sum].flatten()

(deleted_df.groupby('Doors')['Drive wheels'].value_counts() / dd_sum)*100

In [ ]:
deleted_df.groupby('Drive wheels')['Price'].mean().sort_values(ascending=False)

In [ ]:
deleted_df.groupby('Gear box type')['Price'].median().sort_values(ascending=False)

In [ ]:
deleted_df.groupby('Color')['Price'].mean().sort_values(ascending=False)

In [ ]:
deleted_df.groupby('Turbo')['Price'].median().sort_values(ascending=False)

In [ ]:
deleted_df.groupby('Wheel')['Price'].median().sort_values(ascending=False)

In [ ]:
deleted_df.groupby('Fuel type')['Price'].median().sort_values(ascending=False)

In [ ]:
deleted_df['Fuel type'].value_counts()

In [ ]:
%matplotlib inline

deleted_df.hist(bins=25, figsize=(20,15))
plt.show()

As we said at very first from describe method, Price, Mileage, Levy has right skewed distribution and Prod. Year has left skewed distribution.

In [ ]:
attributes = ['Price','Levy','Engine volume','Prod. year','Mileage']
scatter_matrix(deleted_df[attributes], figsize=(12,8))
plt.show()

>📌 There ***positive correlation*** between ***Engine volume and Levy*** and ***slightly negative correlation*** between ***Levy and Prod. year***. Let's check the actual correlation scores between those attributes.

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(deleted_df.corr(),annot=True, cbar = True)
plt.title('Correlation Matrix')
plt.show()

>📌 The assumptions we made from scatter plot seems quite correct! The correliation between Levy and Engine volume ***0.64*** (not bad!). Engine volume and Cylinders has quite strong relationship which is ***0.78*** not that suprising that the cars have more cylinders have more engine volume and Levy.

In [ ]:
num_attribs = ['Levy','Prod. year', 'Engine volume','Doors', 'Mileage', 'Cylinders', 'Airbags']
cat_attribs = ['Manufacturer','Category', 'Leather interior', 'Fuel type', 'Gear box type', 'Drive wheels', 'Wheel', 'Color', 'Turbo']

Let's split our target value from our dataset.

In [ ]:
num_cars = deleted_df[num_attribs]
y = deleted_df['Price']
cat_cars = deleted_df[cat_attribs]

At very first stage we replaced ***'-'*** with ***NaN***s which was for ***Levy*** column. Time to handle with them. We will impute ***median*** values for that NaNs.(We don't have any other NaN value in another column so the following code just will impute them into the Levy but if your dataset has NaN's in the multiple column the code we will type will impute medians to each column seperately.)

In [ ]:
Imputer = SimpleImputer(strategy='median')

Imputer.fit(num_cars)
# Displaying medians of every numveric column we have
display(Imputer.statistics_)
num_cars = Imputer.transform(num_cars)

Time to scale our numerical attributes! Scaling will help to our model to make better predictions and computation time will be less.

In [ ]:
scaler = StandardScaler()
num_cars = scaler.fit_transform(num_cars)

In [ ]:
num_cars

Let's put these 2 steps together to use them with one piece of code when we need them in the future.

In [ ]:
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('std_scaler', StandardScaler())])

Also we need to handle with the categorical columns because the machine learning model we will use expects only numerical values. So we will use one hot encoding, which will give 1 (Hot) the attribute is exist for that sample and 0 (Cold) for others and making full pipeline which will handle numerical values and categorical values at the same time)

In [ ]:
full_pipeline = ColumnTransformer([
    ('num',num_pipeline, num_attribs),
    ('cat',OneHotEncoder(), cat_attribs)  
])
cars_prepared = full_pipeline.fit_transform(deleted_df)

Time to split our data for train and test, we will use %66 of them for train and %33 of them for test step.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(cars_prepared, y, test_size=0.33, random_state = 123)

## Model Setup, Hyperparameter Tuning and Model Evaluation 🧱
<hr>

This is a regression type problem so very first thing I want to try is Linear Regression, ofc!

In [ ]:
lin_reg = LinearRegression()

lin_reg.fit(X_train, y_train)

In [ ]:
predictions = lin_reg.predict(X_train)
lin_mse = mean_squared_error(y_train, predictions)
lin_rmse = np.sqrt(lin_mse)
lin_rmse

We are approximately 9.2k away from actual prices which is not bad let's check MAE as well.

In [ ]:
mae = mean_absolute_error(y_train, predictions)
mae

The next algorithm I wanna try is decision tree, let's check how good it is with that problem.

In [ ]:
tree_reg = DecisionTreeRegressor(random_state = 123)

tree_reg.fit(X_train, y_train)

In [ ]:
tree_predictions = tree_reg.predict(X_train)
tree_mse = mean_squared_error(y_train, tree_predictions)
tree_rmse = np.sqrt(tree_mse)
tree_rmse

Just 995??! Decision tree is really keen on to overfitting so just checking rmse will not be enough to evaluate it let's check cross validation score for it.

In [ ]:
scores = cross_val_score(tree_reg, X_train, y_train, scoring='neg_mean_squared_error', cv=5)

tree_rmse_scores = np.sqrt(-scores)
tree_rmse_scores

OK, the results seems more reasonable now. Our model is doing better than linear regression model.

In [ ]:
lin_scores = cross_val_score(lin_reg, X_train, y_train, scoring='neg_mean_squared_error', cv=5)
lin_rmse_scores = np.sqrt(-lin_scores)
lin_rmse_scores

Another algorithm we will try is Random Forest which is ensemble model. This model relies on randomness, means that n_estimator times decision tree will be trained and the results will combined. (This model will take more time to train than others we tried before as you guess.)

In [ ]:
forest_reg = RandomForestRegressor(n_estimators = 100, random_state=123)

forest_reg.fit(X_train, y_train)

In [ ]:
forest_predictions = forest_reg.predict(X_train)
forest_mse = mean_squared_error(y_train, forest_predictions)
forest_rmse = np.sqrt(forest_mse)
forest_rmse

In [ ]:
forest_scores = cross_val_score(forest_reg, X_train, y_train,
                                scoring="neg_mean_squared_error", cv=5)
forest_rmse_scores = np.sqrt(-forest_scores)
forest_rmse_scores

>❗ Random Forest superiors other 2 algorithm so far so let's continue with this algorithm and tune the hyperparameters for it using GridSearchCV. This step will take 10-20 minutes depending on your computer so be careful before running this cell on your pc)

In [ ]:
param_grid = [
    {'n_estimators': [100, 200], 'max_features': [35,33,31]},
  ]

forest_reg = RandomForestRegressor(random_state=42)

grid_search = GridSearchCV(forest_reg, param_grid, cv=5,
                           scoring='neg_mean_squared_error',
                           return_train_score=True)
grid_search.fit(X_train, y_train)

In [ ]:
grid_search.best_params_

In [ ]:
cvres = grid_search.cv_results_
for mean_score, params in zip(cvres["mean_test_score"], cvres["params"]):
    print(np.sqrt(-mean_score), params)

In [ ]:
feature_importances = grid_search.best_estimator_.feature_importances_
cat_encoder = full_pipeline.named_transformers_["cat"]
cat_one_hot_attribs = [i  for cat in cat_encoder.categories_ for i in cat]
attributes = num_attribs +  cat_one_hot_attribs
sorted(zip(feature_importances, attributes), reverse=True)

In [ ]:
final_model = grid_search.best_estimator_
final_predictions = final_model.predict(X_test)
final_mse = mean_squared_error(y_test, final_predictions)
final_rmse = np.sqrt(final_mse)
final_rmse

Sooo as a final result we are 5.7k far away from actual results, this is not bad! Also we seems like we don't have overfitting problem at all, means that our model generalizes well. Let's make confidence interval to serve our results.

In [ ]:
confidence = 0.95
squared_errors = (final_predictions - y_test) ** 2
np.sqrt(stats.t.interval(confidence, len(squared_errors) - 1,
                         loc=squared_errors.mean(),
                         scale=stats.sem(squared_errors)))

We are 95 percent confident to say that our errors will be in the range of (5506.64790839 , 6004.65750968).

In [ ]:
# Save the model we trained
joblib.dump(final_model, "final_model.pkl")

# If you want to use this model all you need to do is:
# joblib.load('final_model.pkl')

## Extra

We checked the feature importances couple of cell before and some of the features seem not adding quite information to our model. So let's pick first k features to train our model. In order to do that, we will make custom transformer and will use it in the Pipeline which will take care data cleaning, selection of top k features and training our model.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

def indices_of_top_k(arr, k):
    return np.sort(np.argpartition(np.array(arr), -k)[-k:])

class TopFeatureSelector(BaseEstimator, TransformerMixin):
    def __init__(self, feature_importances, k):
        self.feature_importances = feature_importances
        self.k = k
    def fit(self, X, y=None):
        self.feature_indices_ = indices_of_top_k(self.feature_importances, self.k)
        return self
    def transform(self, X):
        return X[:, self.feature_indices_]

In [ ]:
final_pipeline = Pipeline([('full',full_pipeline),
                           ('top_feature_selector',TopFeatureSelector(feature_importances, 35)),
                          ('model', final_model)])

In [ ]:
final_pipeline.fit(deleted_df.drop('Price', axis=1), deleted_df['Price'])

In [ ]:
some_data = deleted_df.drop('Price', axis=1).iloc[:4]
some_labels = deleted_df['Price'].iloc[:4]

pred = final_pipeline.predict(some_data)
display(pred)
display(some_labels.values)


<center><img
src="https://monophy.com/media/l0HlRaCJRVkRiVUNa/monophy.gif" style="width:70%;height:70%;">
</center>

This is the end of our notebook! We've done lots of stuff data cleaning, model training, evaluation of model and many more. Looking forward for your thoughts in the comment section. Peace out :)